In [1]:
using LinearAlgebra, Plots, Distributions, LaTeXStrings, Random, Optim, TracyWidomBeta, DataFrames, CSV

ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.


In [ ]:
include(joinpath(@__DIR__, "..", "..", "AuxiliaryFunctions.jl"))
include(joinpath(@__DIR__, "..", "..", "SpikeEstimation.jl"))
include(joinpath(@__DIR__, "..", "..", "BEMA.jl"))
include(joinpath(@__DIR__, "..", "..", "PA.jl"))
include(joinpath(@__DIR__, "..", "..", "PassYao.jl"))
figdir = "Figures"
tbdir = "Tables"

PassYao (generic function with 1 method)

In [ ]:
Random.seed!(1234)

N = 6000
d = 0.1
M = convert(Int64,ceil(N/d))
X = randn(N,M)

K = 200
nodes, weights = LegQuad(K)
a = 0.1; b = 4.0
h = x-> a<x<b ? (2*(3.5-x)^3+x)*(b-x)^(1/2)*(x-a)^(1/2)/(2*(4.5-x)^2) : 0
normCst = LegQuadInt(h,a,b,nodes,weights)
scaled_h = x->h(x)/normCst
quantiles = zeros(Float64,N+1)
quantiles[1] = a
quantiles[N+1] = b
for i=2:N
    QuantEq = x->LegQuadInt(scaled_h,a,x,nodes,weights)-(i-1)/N
    quantiles[i] = Bisection(QuantEq,quantiles[1],quantiles[N+1])
end

δ = 6
quantiles[1:2] = [7,δ]
sqrtΣ = Diagonal(sqrt.(quantiles[1:end-1]))
W = Matrix{Float64}(undef,N,N)
rmul!(X, inv(sqrt(M)))
mul!(X,sqrtΣ,X)
mul!(W,X,X')
evals = eigvals(Symmetric(W))

true_spikes = evals[end:-1:end-1]
p1 = histogram(quantiles,bins=quantiles[3]-0.2:0.1:quantiles[1]+0.2,normalize=:pdf,label="ESD of Σ",framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
color = theme_palette(:auto).colors[1]
p1 = scatter!(quantiles[1:2],0*quantiles[1:2],markersize=8,color=color,marker=:dot,label="Spikes of Σ",alpha=0.5)

results = AsympSCM(W, vecNbr=100, nx=1000, tol=3/sqrt(N))
xvec, yvec, SpikeNbr, SpikeLoc, γmin, γplus = results[:x], results[:density], results[:spikes_nbr], results[:spikes_loc], results[:γmin], results[:γplus]

p2 = histogram(evals,bins=γmin-0.2:0.1:γplus+0.2,normalize=:pdf,label="ESD of "*L"W",framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
color = theme_palette(:auto).colors[1]
p2 = plot!(xvec,yvec,linecolor=:red,linewidth=4,label="Estimated density")
p2 = scatter!(true_spikes,0*true_spikes,markersize=8,color=color,marker=:dot,label="True Outliers")
p2 = scatter!(SpikeLoc,0*SpikeLoc,markersize=5,color=:red,marker=:dot,label="Estimated Outliers")
savefig(p1,joinpath(figdir, "Sigma.pdf"))
savefig(p2,joinpath(figdir, "Density.pdf"))
tb = DataFrame(A=true_spikes,B=SpikeLoc,C=abs.(true_spikes-SpikeLoc))
CSV.write(joinpath(tbdir, "Spikes.csv"),tb)

"Spikes.csv"

In [ ]:
Random.seed!(1234)

function process_sample(N, M, d, X, sqrtΣ, W, SpikeNbr, t, MPquants)
    locPercent = zeros(Float64,5)
    locAvrg = zeros(Float64,5)
    locTime = zeros(Float64,5)
    randn!(X)
    rmul!(X, inv(sqrt(M)))
    mul!(X,sqrtΣ,X)
    mul!(W,X,X')       
    time_evals = @elapsed begin
        evals = eigvals(Symmetric(W))
    end
    time_BEMA0 = @elapsed begin
        BEMA0Out = BEMA0(evals,d, t, MPquants)
        locPercent[1] = BEMA0Out==SpikeNbr ? 1 : 0
        locAvrg[1] = BEMA0Out
    end
    time_BEMA = @elapsed begin
        BEMAOut = BEMA(evals,d;SampleNbr=10)
        locPercent[2] = BEMAOut==SpikeNbr ? 1 : 0
        locAvrg[2] = BEMAOut
    end
    time_PassYao = @elapsed begin
        PassYaoOut = PassYao(evals,d,N;SampleNbr=15)
        locPercent[3] = PassYaoOut==SpikeNbr ? 1 : 0
        locAvrg[3] = PassYaoOut
    end
    time_DDPA = @elapsed begin
        DDPAOut = DDPA(evals,N,d)
        locPercent[4] = DDPAOut==SpikeNbr ? 1 : 0
        locAvrg[4] = DDPAOut
    end
    time_CholList = @elapsed begin
        results = AsympSCM(W, vecNbr=1, tol = 3/sqrt(N), compute_density = false)
        Nbr = results[:spikes_nbr]
        locPercent[5] = Nbr==SpikeNbr ? 1 : 0
        locAvrg[5] = Nbr
    end
    locTime[1] = time_evals+time_BEMA0
    locTime[2] = time_evals+time_BEMA
    locTime[3] = time_evals+time_PassYao
    locTime[4] = time_evals+time_DDPA
    locTime[5] = time_CholList
    return (locPercent,locAvrg,locTime)
end

## Warm up

N = 1000
d = 0.5
M = convert(Int64,ceil(N/d))
X = randn(Float64,N,M)

K = 200
nodes, weights = LegQuad(K)
a = 0.1; b = 4.0
h = x-> a<x<b ? (2*(3.5-x)^3+x)*(b-x)^(1/2)*(x-a)^(1/2)/(2*(4.5-x)^2) : 0
normCst = LegQuadInt(h,a,b,nodes,weights)
scaled_h = x->h(x)/normCst
quantiles = zeros(Float64,N+1)
quantiles[1] = a
quantiles[N+1] = b
for i=2:N
    QuantEq = x->LegQuadInt(scaled_h,a,x,nodes,weights)-(i-1)/N
    quantiles[i] = Bisection(QuantEq,quantiles[1],quantiles[N+1])
end
sqrtΣ = Diagonal(sqrt.(quantiles[1:end-1]))
W = Matrix{Float64}(undef,N,N)

t = TWquant(0.1)
MPquants = quantMP(N,d)

sqrtΣ[1,1] = sqrt(7); sqrtΣ[2,2] = sqrt(5)
process_sample(N, M, d, X, sqrtΣ, W, 2, t, MPquants)

## Main

N = 3000
d = 0.1
M = convert(Int64,ceil(N/d))
X = randn(Float64,N,M)

K = 200
nodes, weights = LegQuad(K)
a = 0.1; b = 4.0
h = x-> a<x<b ? (2*(3.5-x)^3+x)*(b-x)^(1/2)*(x-a)^(1/2)/(2*(4.5-x)^2) : 0
normCst = LegQuadInt(h,a,b,nodes,weights)
scaled_h = x->h(x)/normCst
quantiles = zeros(Float64,N+1)
quantiles[1] = a
quantiles[N+1] = b
for i=2:N
    QuantEq = x->LegQuadInt(scaled_h,a,x,nodes,weights)-(i-1)/N
    quantiles[i] = Bisection(QuantEq,quantiles[1],quantiles[N+1])
end
sqrtΣ = Diagonal(sqrt.(quantiles[1:end-1]))
W = Matrix{Float64}(undef,N,N)

t = TWquant(0.1)
MPquants = quantMP(N,d)

δvec = [5,7,9,21]
lenδ = length(δvec)
SampleNbr = 50
SpikeNbr = 2
Percent = zeros(Float64,lenδ,5)
Avrg = zeros(Float64,lenδ,5)
Time = zeros(Float64,lenδ,5)

for i=1:lenδ
    δ = δvec[i]
    sqrtΣ[1,1] = sqrt(7)
    sqrtΣ[2,2] = sqrt(δ)
    results = map(_-> process_sample(N, M, d, X, sqrtΣ, W, SpikeNbr, t, MPquants), 1:SampleNbr)
    locPercent = first.(results)
    locAvrg = getindex.(results, 2)
    locTime = getindex.(results, 3)

    Percent[i,:] = vec(reduce( .+, locPercent))/SampleNbr
    Avrg[i,:] = vec(reduce( .+, locAvrg))/SampleNbr
    Time[i,:] = vec(reduce( .+, locTime))/SampleNbr


    tb = DataFrame(A=δvec[1:i],B=Percent[1:i,1],C=Percent[1:i,2],D=Percent[1:i,3],E=Percent[1:i,4],F=Percent[1:i,5])
    CSV.write(joinpath(tbdir, "Percent.csv"),tb)
    tb = DataFrame(A=δvec[1:i],B=Avrg[1:i,1],C=Avrg[1:i,2],D=Avrg[1:i,3],E=Avrg[1:i,4],F=Avrg[1:i,5])
    CSV.write(joinpath(tbdir, "Avrg.csv"),tb)
    tb = DataFrame(A=δvec[1:i],B=Time[1:i,1],C=Time[1:i,2],D=Time[1:i,3],E=Time[1:i,4],F=Time[1:i,5])
    CSV.write(joinpath(tbdir, "Time.csv"),tb)
end